# expenses_income_summary

In [1]:
import pandas as pd
import numpy as np

df7 = pd.read_csv("../Datasets/expenses_income_summary.csv")
print("[INFO] Loaded:", df7.shape)
df7.head()

[INFO] Loaded: (1155, 15)


,Date,title,category,account,amount,currency,type,transfer-amount,transfer-currency,to-account,receive-amount,receive-currency,description,due-date,id
0,2024-08-11 13:56:59.652,Karthik,Bills & Fees,Savings Bank,45.00,INR,EXPENSE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,74e78631-db14-4495-bfb9-85546b0bd2fe
1,2024-08-10 16:09:55.986,Juice,Food & Drinks,Cash,40.00,INR,EXPENSE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65e12e62-9f63-4c6c-b452-6c7b42fbfb7f
2,2024-08-09 10:25:21.618,Tire,Transport,Cash,10.00,INR,EXPENSE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9ecd93bd-a835-4263-86e2-99fea475fa37
3,2024-08-07 03:57:24.944,Baba,Bills & Fees,Savings Bank,200.00,INR,EXPENSE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,00d39b2c-e722-485a-85ca-28f6506dc674
4,2024-08-04 13:09:08.452,Reward,Bills & Fees,Salary Bank,4.00,INR,INCOME,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3861d205-3245-4926-ad69-4491b0bff547


Drop Unnecessary Columns

In [2]:
drop_cols = [
    "transfer-amount", "transfer-currency", "to-account",
    "receive-amount", "receive-currency", "description",
    "due-date", "id"
]
df7 = df7.drop(columns=drop_cols, errors="ignore")

Check missing values 

In [3]:
print("\n[INFO] Missing Values:")
print(df7.isna().sum())


[INFO] Missing Values:
Date          0
title        24
category    161
account       0
amount        0
currency      0
type          0
dtype: int64


Check for Duplicates 

In [4]:
dup_count = df7.duplicated().sum()
print("\n[INFO] Duplicate rows:", dup_count)


[INFO] Duplicate rows: 0


Standardize Column Names

In [5]:
df7 = df7.rename(columns={
    "Date": "Date",
    "title": "Title",
    "category": "Category",
    "account": "Account Name",
    "amount": "Amount",
    "currency": "Currency",
    "type": "Type"
})

Fix DATE Column

In [6]:
df7["Date"] = pd.to_datetime(df7["Date"], errors="coerce").dt.date
print("\n[INFO] Invalid dates:", df7["Date"].isna().sum())


[INFO] Invalid dates: 0


Clean AMOUNT Column

In [7]:
print("\n[INFO] Checking invalid amount formats...")

# Step 1: Clean commas for checking
clean_amount = (
    df7["Amount"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
)

# Step 2: Identify invalid numeric formats
invalid_amounts = df7[
    ~clean_amount.str.replace(".", "", 1).str.isnumeric()
]

print("[INFO] Invalid examples:")
print(invalid_amounts.head())


[INFO] Checking invalid amount formats...
[INFO] Invalid examples:
Empty DataFrame
Columns: [Date, Title, Category, Account Name, Amount, Currency, Type]
Index: []


In [8]:
# Keep only numeric-formatted amounts
df7 = df7[
    clean_amount.str.replace(".", "", 1).str.isnumeric()
].copy()

# Convert Amount to float (after removing commas)
df7["Amount"] = (
    df7["Amount"]
    .astype(str)
    .str.replace(",", "", regex=False)
)

df7["Amount"] = pd.to_numeric(df7["Amount"], errors="coerce")

In [9]:
# Drop non-positive values
df7 = df7[df7["Amount"] > 0]

FIX MISSING CATEGORY + TITLE

In [10]:
df7["Category"] = df7["Category"].fillna("Other")
df7["Title"] = df7["Title"].fillna("Unknown")

df7["Category"] = df7["Category"].astype(str).str.strip().str.title()
df7["Title"] = df7["Title"].astype(str).str.strip().str.title()

CREATE TransactionID

In [11]:
df7 = df7.reset_index(drop=True)
df7["TransactionID"] = df7.index + 1
df7["TransactionID"] = df7["TransactionID"].apply(
    lambda x: f"TRX7_{str(x).zfill(6)}"
)

ASSIGN USER ID

In [12]:
df7["UserID"] = "US7000"

In [13]:
# Add empty Merchant column
df7["Merchant"] = ""

LLM: Generate Transaction Description

In [14]:
from groq import Groq

client = Groq(api_key="YOUR_GROQ_API_KEY")

def generate_description(title, category, txn_type):
    prompt = f"""
Generate a short, realistic bank transaction description using ONLY:
- Title: {title}
- Category: {category}
- Type: {txn_type}

Rules:
- Simple natural sentence
- 6 to 12 words
- No merchant names
- No hallucination
- No numbers
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=30,
        temperature=0.4
    )
    return response.choices[0].message.content.strip()

In [15]:
# Generate once per title
unique_titles = df7["Title"].unique()
desc_map = {}

for t in unique_titles:
    row = df7[df7["Title"] == t].iloc[0]
    desc_map[t] = generate_description(
        t,
        row["Category"],
        row["Type"]
    )

df7["Transaction Description"] = df7["Title"].map(desc_map)

AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

DROP TITLE COLUMN (NO LONGER NEEDED)

In [ ]:
df7 = df7.drop(columns=["Title"], errors="ignore")

FINAL VALIDATION

In [ ]:
required = ["Date", "Category", "Amount", "UserID", "TransactionID"]

print("\n[INFO] Missing required fields:")
print(df7[required].isna().sum())

print("\n[INFO] Final shape:", df7.shape)


In [ ]:
df7.head()

SAVE CLEANED DATASET

In [ ]:
import os
os.makedirs("Tofinal", exist_ok=True)

df7.to_csv("Tofinal/expenses_income_summary.csv", index=False)
print("\nFile saved → Tofinal/expenses_income_summary.csv")